In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy import interpolate
import matplotlib.pyplot as plt
from scipy.signal import resample
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [5]:
# root dir
root = 'Depression/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,Original_ID,sex,age,BDI,STAI,SCID,SCID_notes,HamD
0,sub-001,507,1.0,19.0,0.0,23.0,No Interview,NaN,NaN
1,sub-002,508,1.0,18.0,4.0,47.0,No Interview,NaN,NaN
2,sub-003,509,1.0,18.0,7.0,44.0,No Interview,NaN,NaN
3,sub-004,510,1.0,19.0,1.0,27.0,No Interview,NaN,NaN
4,sub-005,511,2.0,22.0,1.0,23.0,No Interview,NaN,NaN
...,...,...,...,...,...,...,...,...,...
117,sub-118,624,1.0,20.0,23.0,60.0,Current MDD,NaN,21.0
118,sub-119,625,1.0,19.0,16.0,60.0,Past MDD,subsyndromal current,4.0
119,sub-120,626,1.0,18.0,14.0,41.0,Current MDD,NaN,10.0
120,sub-121,627,2.0,19.0,30.0,47.0,Past MDD,NaN,3.0


## Features

In [6]:
# Test for bad channels, sampling freq and shape
"""bad_channel_list, sampling_freq_list, data_shape_list = [], [], []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                # print(raw.get_montage())
                # get bad channels
                bad_channel = raw.info['bads']
                bad_channel_list.append(bad_channel)
                # get sampling frequency
                sampling_freq = raw.info['sfreq']
                sampling_freq_list.append(sampling_freq)
                # get eeg data
                data = raw.get_data()
                data_shape = data.shape
                data_shape_list.append(data_shape)"""

"bad_channel_list, sampling_freq_list, data_shape_list = [], [], []\nfor sub in os.listdir(root):\n    if 'sub-' in sub:\n        sub_path = os.path.join(root, sub, 'eeg/')\n        # print(sub_path)\n        for file in os.listdir(sub_path):\n            if '.set' in file:\n                file_path = os.path.join(sub_path, file)\n                raw = mne.io.read_raw_eeglab(file_path, preload=True)\n                # print(raw.get_montage())\n                # get bad channels\n                bad_channel = raw.info['bads']\n                bad_channel_list.append(bad_channel)\n                # get sampling frequency\n                sampling_freq = raw.info['sfreq']\n                sampling_freq_list.append(sampling_freq)\n                # get eeg data\n                data = raw.get_data()\n                data_shape = data.shape\n                data_shape_list.append(data_shape)"

In [7]:
"""from collections import Counter

print(bad_channel_list)
print(data_shape_list[0])
print("Channel number counter:", Counter(i[0] for i in data_shape_list))
print("Sampling rate counter:", Counter(sampling_freq_list))"""

'from collections import Counter\n\nprint(bad_channel_list)\nprint(data_shape_list[0])\nprint("Channel number counter:", Counter(i[0] for i in data_shape_list))\nprint("Sampling rate counter:", Counter(sampling_freq_list))'

## Pick common channels

In [8]:
# channel number not consistent, take the common channels
"""common_channels = []
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        # print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                current_channels = set(raw.info['ch_names'])
                if not common_channels:
                    common_channels = current_channels
                else:
                    common_channels &= current_channels
common_channels = list(common_channels)
print(common_channels)
print("Common channels number: ", len(common_channels))"""

'common_channels = []\nfor sub in os.listdir(root):\n    if \'sub-\' in sub:\n        sub_path = os.path.join(root, sub, \'eeg/\')\n        # print(sub_path)\n        for file in os.listdir(sub_path):\n            if \'.set\' in file:\n                file_path = os.path.join(sub_path, file)\n                raw = mne.io.read_raw_eeglab(file_path, preload=True)\n                current_channels = set(raw.info[\'ch_names\'])\n                if not common_channels:\n                    common_channels = current_channels\n                else:\n                    common_channels &= current_channels\ncommon_channels = list(common_channels)\nprint(common_channels)\nprint("Common channels number: ", len(common_channels))'

In [9]:
common_channels = ['T7', 'PZ', 'F5', 'PO3', 'PO6', 'C4', 'TP7', 'CZ', 'O2', 'CP4', 'F2', 'PO5', 'O1', 'CP3', 'PO4', 'P5', 'F8', 'PO8', 'P8', 'F7', 'POZ', 'FC5', 'OZ', 'FC2', 'CPZ', 'C6', 'C1', 'TP8', 'AF3', 'CP5', 'P4', 'P3', 'FP2', 'FZ', 'FP1', 'HEOG', 'CP6', 'P1', 'FT8', 'C5', 'F3', 'F6', 'FC3', 'C2', 'P2', 'PO7', 'FC1', 'FT7', 'CP1', 'F1', 'P7', 'CP2', 'F4', 'P6', 'C3', 'AF4', 'M1', 'VEOG', 'FCZ', 'FC4', 'FPZ', 'CB2', 'FC6', 'M2', 'T8', 'CB1']

## Data preprocessing

In [10]:
def data_preprocessing(
    raw: mne.io.Raw,
    common_channels: list,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    do_bad_interp: bool = True,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      1) choose common channels and reorder
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
    
    # 1. select common channels
    raw.pick_channels(common_channels)
    if verbose:
        print(f"✔ Step 1: Picked common channels ({len(common_channels)}): {common_channels}")
        
    # non-readable channels: ['FP1', 'FPZ', 'FP2', 'FZ', 'FCZ', 'CZ', 'CPZ', 'PZ', 'POZ', 'CB1', 'OZ', 'CB2']
    
    # Since current channels are named in capital letters, we need to rename them to match the standard montage. 
    # Let's manually set it for simplicity, and remove the non-eeg channels.
    ch_map = {'T7':'T7', 'PZ':'Pz', 'F5':'F5', 'PO3':'PO3', 'PO6':'PO6', 'C4':'C4', 'TP7':'TP7', 'CZ':'Cz', 'O2':'O2', 'CP4':'CP4', 'F2':'F2', 'PO5':'PO5', 'O1':'O1', 'CP3':'CP3', 'PO4':'PO4', 'P5':'P5', 'F8':'F8', 'PO8':'PO8', 'P8':'P8', 'F7':'F7', 'POZ':'POz', 'FC5':'FC5', 'OZ':'Oz', 'FC2':'FC2', 'CPZ':'CPz', 'C6':'C6', 'C1':'C1', 'TP8':'TP8', 'AF3':'AF3', 'CP5':'CP5', 'P4':'P4', 'P3':'P3', 'FP2':'Fp2', 'FZ':'Fz', 'FP1':'Fp1', 'CP6':'CP6', 'P1':'P1', 'FT8':'FT8', 'C5':'C5', 'F3':'F3', 'F6':'F6', 'FC3':'FC3', 'C2':'C2', 'P2':'P2', 'PO7':'PO7', 'FC1':'FC1', 'FT7':'FT7', 'CP1':'CP1', 'F1':'F1', 'P7':'P7', 'CP2':'CP2', 'F4':'F4', 'P6':'P6', 'C3':'C3', 'AF4':'AF4', 'M1':'M1', 'FCZ':'FCz', 'FC4':'FC4', 'FPZ':'Fpz', 'FC6':'FC6', 'M2':'M2', 'T8':'T8'}
    # rename the channels 
    raw.rename_channels(ch_map)
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    # 5. Interpolate bad channels
    if do_bad_interp and raw.info.get("bads"):
        raw.interpolate_bads(reset_bads=True, verbose=False)
        if verbose:
            print(f"✔ Step 5: Interpolated bads: {raw.info.get('bads', [])}")
    else:
        if verbose:
            print("ℹ Step 5: No bads to interpolate (set raw.info['bads'] first if needed)")
            
    # 6) Re-reference to average
    raw.set_eeg_reference("average", verbose=False)
    if verbose:
        print("✔ Step 6: Average reference")
    
    # 5. ICA + ICLabel
    raw_ica = raw.copy()
    ica = ICA(n_components=0.99, random_state=97, max_iter='auto')
    ica.fit(raw_ica)
    if verbose:
        print("✔ ICA fitted.")

    try:
        ic_labels = label_components(raw_ica, ica, method='iclabel')
        labels = ic_labels['labels']
        probs = ic_labels['y_pred_proba']

        artifact_thresholds = {
            'eye blink': 0.7,
            'muscle artifact': 0.6,
            'heart beat': 0.5,
            'line noise': 0.8,
            'channel noise': 0.9
        }

        to_exclude = [
            i for i, label in enumerate(labels)
            if label in artifact_thresholds and probs[i] >= artifact_thresholds[label]
        ]
        if to_exclude:
            ica.exclude = to_exclude
            ica.apply(raw)
            if verbose:
                print(f"✔ ICA applied. Excluded components: {to_exclude}")
        else:
            if verbose:
                print("No ICs exceeded artifact thresholds. No components excluded.")

    except Exception as e:
        if verbose:
            print(f"⚠ ICLabel failed: {e}. Proceeding without ICA-based removal.")
        
    return raw

In [11]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [13]:
# Loop through each subject and session to preprocess the EEG data
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
# 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
# For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6
# So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP, "\n")
sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                eog_channels = ['HEOG', 'VEOG']  # EOG channels
                unkown_channels = ['CB1', 'CB2']  # Unknown channels
                # remove eog channels and unknown channels
                common_channels = list(set(common_channels) - set(eog_channels) - set(unkown_channels))
                raw.pick(common_channels[:19])  # pick first 19 channels, as we only need 19 channels for counting segments
                print("Output channels order: ", raw.info['ch_names'])
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
                data = raw.get_data().T
                for fs in SAMPLE_RATE_LIST:
                    T_res = int(T_raw * fs / original_fs)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub_id][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
        sub_id += 1
    print("-------------------------------------\n")


print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200 

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

Depression/sub-001\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\Depression\Depression\sub-001\eeg\sub-001_task-Rest_run-01_eeg.fdt
Reading 0 ... 250733  =      0.000 ...   501.466 secs...
Output channels order:  ['FP2', 'F6', 'M2', 'FPZ', 'AF4', 'PO3', 'F7', 'T7', 'P1', 'F3', 'FC1', 'AF3', 'CP5', 'M1', 'C2', 'FC5', 'CP4', 'O2', 'CP3']
fs=200Hz: T_res=100293, STEP=200, n_seg=500
fs=100Hz: T_res=50146, STEP=200, n_seg=249
fs=50Hz: T_res=25073, STEP=200, n_seg=124
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\Depression\Depression\sub-001\eeg\sub-001_task-Rest_run-02_eeg.fdt
Reading 0 ... 184092  =      0.000 ...   368.184 secs...
Output channels orde

In [14]:
output_root = os.path.join('Processed', sub_folder_path, 'Depression')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\Depression\X.dat
y path: Processed\L400\Depression\y.dat


## PASS 2: Process and store segments into memmap

In [15]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        li_sub = []
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        sub_label = 0  # we use for pretrain, so all label=0 is fine here for simplicity
        for file in os.listdir(sub_path):
            if '.set' in file:
                file_path = os.path.join(sub_path, file)
                raw = mne.io.read_raw_eeglab(file_path, preload=True)
                eog_channels = ['HEOG', 'VEOG']  # EOG channels
                unkown_channels = ['CB1', 'CB2']  # Unknown channels
                # remove eog channels and unknown channels
                common_channels = list(set(common_channels) - set(eog_channels) - set(unkown_channels))
                raw = data_preprocessing(
                    raw=raw,
                    common_channels=common_channels,  # common channels
                    notch_freq=60,  # data collected in United States, so notch at 60 Hz
                    l_freq=0.5,
                    h_freq=45.0,
                    do_bad_interp=True,
                    verbose=True
                )
                # 19 standard channels should be: 'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3', 'Pz', 'P4', 'T6', 'O1', 'O2'
                # For here, T7, T8 is the same to T3, T4; P7, P8 is the same to T5, T6;
                # So we use T7, T8, P7, P8 to replace T3, T4, T5, T6
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                raw.pick(standard_channels)
                print("Output channels order: ", raw.info['ch_names'])
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
                data = raw.get_data().T
                total_seconds_all += data.shape[0] / original_fs
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data_resampled = resample_time_series(data, original_fs, fs)
                    T_res, _ = data_resampled.shape
                    print(f"fs={fs}Hz: resampled shape={data_resampled.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data_resampled[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(sub_label)       # label
                        y_mm[cur, 1] = float(sub_id)      # Global trial ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
        sub_id += 1
    print("-------------------------------------\n")


total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

-------------------------------------

Depression/sub-001\eeg/
Reading D:\PycharmPorjects\DataProcessingLocalDisk\LEAD-4\Depression\Depression\sub-001\eeg\sub-001_task-Rest_run-01_eeg.fdt
Reading 0 ... 250733  =      0.000 ...   501.466 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
✔ Step 1: Picked common channels (62): ['FT7', 'PO4', 'FP2', 'F6', 'M2', 'O1', 'FPZ', 'AF4', 'F5', 'POZ', 'FC6', 'P5', 'PO3', 'F7', 'T7', 'PZ', 'F3', 'FC1', 'M1', 'AF3', 'CP5', 'P1', 'PO5', 'CP2', 'C2', 'F2', 'CPZ', 'FC5', 'CP4', 'O2', 'T8', 'C5', 'C4', 'FP1', 'F4', 'P8', 'FT8', 'CP3', 'OZ', 'FC4', 'PO8', 'PO7', 'PO6', 'FZ', 'C3', 'CZ', 'P6', 'FC2', 'FC3', 'P2', 'C6', 'F1', 'C1', 'FCZ', 'P3', 'TP7', 'TP8', 'CP1', 'F8', 'P4', 'P7', 'CP6']

## Load and check the processed data

In [16]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 146062
  T = 400
  C = 19
  X_path = Processed\L400\Depression\X.dat
  y_path = Processed\L400\Depression\y.dat
-------------------------------------
Subject ID 001: X shape=(1514, 400, 19), y shape=(1514, 3)
Subject ID 002: X shape=(1489, 400, 19), y shape=(1489, 3)
Subject ID 003: X shape=(1587, 400, 19), y shape=(1587, 3)
Subject ID 004: X shape=(1234, 400, 19), y shape=(1234, 3)
Subject ID 005: X shape=(1240, 400, 19), y shape=(1240, 3)
Subject ID 006: X shape=(1202, 400, 19), y shape=(1202, 3)
Subject ID 007: X shape=(1142, 400, 19), y shape=(1142, 3)
Subject ID 008: X shape=(1139, 400, 19), y shape=(1139, 3)
Subject ID 009: X shape=(1209, 400, 19), y shape=(1209, 3)
Subject ID 010: X shape=(869, 400, 19), y shape=(869, 3)
Subject ID 011: X shape=(1206, 400, 19), y shape=(1206, 3)
Subject ID 012: X shape=(1198, 400, 19), y shape=(1198, 3)
Subject ID 013: X shape=(1237, 400, 19), y shape=(1237, 3)
Subject ID 014: X shape=(1154, 400, 19), y shape=(1154, 3)
Subject ID 015